# Modeling experimentation

As mentioned in the recommendation section of the README.md and Modeling.ipynb we are going to test a few models and a bit of hyperparameter tuning and some class balancing. We will start with the models, then tune and finally class balance before running the models one more time to see if we can further improve the model.

We will test the data on XGBoost as it often achieves higher predictive accuracy than Random Forest on tabular data because it builds its trees sequentially, correcting the previous trees errors as it progresses through the modeling process. This can yield a better ROC-AUC and more precise probability estimates. 

It is important to note that higher accuracy may not lead to a higher open rate for our client. The open rate achieved by targeting is determined by how well the model ranks customers. If Gradient boosting improves the ranking then reach for a given send volume will increase. So improved model performance should lead to higher open rate among the targeted group, if all else remains the same.

Using the exact same data and model as Modeling.ipynb notebook and adding XGBoost to compare side by side.

In [1]:
# Add to venv if wishing to experiment
# !pip install xgboost

In [2]:
# !pip install mlflow

In [3]:
# Library imports
import pandas as pd
import mlflow
import mlflow.sklearn
import numpy as np
from sklearn.model_selection import GridSearchCV,  train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

#### Generate Synthetic Data and Target Variable

In [4]:
# Random seed for reproducibility
np.random.seed(42)

# Generating synthetic customer data
n_customers = 5000
data = {
    'customer_id': range(1, n_customers+1),
    'age' : np.random.randint(18, 70, n_customers),
    'income': np.random.normal(50000, 15000, n_customers).astype(int), # estimated
    'tenure': np.random.randint(1, 120, n_customers), # months
    'last_purchase_days': np.random.randint(1, 365, n_customers),
    'avg_order_value': np.random.normal(100, 30, n_customers).clip(20, 300).astype(int),
    'campaign_channel': np.random.choice(['email','social', 'push'], n_customers),
    'campaign_type': np.random.choice(['promotional', 'informational', 'loyalty'], n_customers),
    'sent': np.random.choice([0,1], n_customers, p=[0.2,0.8]), # 80% were sent
}

df = pd.DataFrame(data)

In [5]:
# Generate target variable 'opened' based on some hidden pattern
# Higher probability if: recent purchases, higher income, email channel, promotional type
def generate_opened(row):
    base_prob = 0.1
    if row['last_purchase_days'] > 60:
        base_prob += 2.0
    if row['income'] > 60000:
        base_prob += 1.0
    if row['campaign_channel'] == 'email':
        base_prob += 0.15
    if row['campaign_type'] == 'promotional':
        base_prob += 0.1
    # cap probability
    prob = min(base_prob, 0.9)
    return np.random.binomial(1, prob)

df['opened'] = df.apply(generate_opened, axis =1)

print("Data shape:", df.shape)
df.head()

Data shape: (5000, 10)


,customer_id,age,income,tenure,last_purchase_days,avg_order_value,campaign_channel,campaign_type,sent,opened
0,1,56,48353,96,249,95,email,loyalty,0,1
1,2,69,57462,92,74,146,push,promotional,1,1
2,3,46,44219,48,319,130,email,loyalty,1,1
3,4,32,56306,39,296,73,social,loyalty,1,1
4,5,60,37034,40,274,82,push,promotional,1,1


## Feature Engineering, Preprocessing and Modeling Pipelines

In [6]:
# Using only customers who were sent the campaign
train_df = df[df['sent']==1].copy()

# Features for the model
features = ['age', 'income', 'tenure', 'last_purchase_days', 'avg_order_value', 'campaign_channel', 'campaign_type']
X = train_df[features]
y = train_df['opened']

#  Splitting into train and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Preprocessing: scale numeric, one-hot encode categorical
numeric_features = ['age', 'income', 'tenure', 'last_purchase_days', 'avg_order_value']
numeric_transformer = StandardScaler()
categorical_features = ['campaign_channel', 'campaign_type']
categorical_transformer = OneHotEncoder(drop='first', handle_unknown = 'ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

modelRF = Pipeline(steps= [
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

modelXG = Pipeline(steps= [
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(n_estimators=100, random_state=42))
])

# Train with MLflow tracking
with mlflow.start_run(run_name="Random Forest with Preprocessing"):
    modelRF.fit(X_train, y_train)
    y_proba = modelRF.predict_proba(X_test)[:,1]
    auc = roc_auc_score(y_test, y_proba)
    mlflow.log_metric("roc_auc", auc)
    mlflow.sklearn.log_model(modelRF, "model")
    print(f"ROC-AUC RF: {auc}")

with mlflow.start_run(run_name="XGBoost"):
    modelXG.fit(X_train, y_train)
    y_proba = modelXG.predict_proba(X_test)[:,1]
    auc = roc_auc_score(y_test, y_proba)
    mlflow.log_metric("roc_auc", auc)
    mlflow.sklearn.log_model(modelXG, "model")
    print(f"ROC-AUC XG: {auc}")

2026/03/30 13:27:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/30 13:27:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/30 13:27:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/30 13:27:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

ROC-AUC RF: 0.781379487179487
ROC-AUC XG: 0.7818871794871796


## Performing GridSearchCV to find the best model and its parameters

MLflow can automatically log parameters and metrics for sklearn models including those from grid search CV.

In [7]:
# Pipelines for each of the GridSearches

# Pipelines
pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

pipeline_xgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(random_state=42, eval_metric='logloss'))
])

In [8]:
# Parameter grids
# Parameter grid for Random Forest
param_grid_rf = {
    'classifier__n_estimators': [50, 75, 100, 150],
    'classifier__max_depth': [None, 20, 30, 40],
    'classifier__min_samples_split': [8, 10, 12, 15], 
    'classifier__class_weight': [
        None, 
        'balanced',                    # automatically adjust weights inversely to class frequencies
        {0: 2.0, 1: 1.0},             # weight minority class (0) higher
        {0: 1.5, 1: 1.0},
        {0: 1.33, 1: 1.0}
    ] 
}

# Parameter grid for XGBoost
param_grid_xgb = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [3, 6, 9],
    'classifier__learning_rate': [0.01, 0.1, 0.2],
    'classifier__subsample': [0.8, 1.0]
    # Note: XGBoost's 'learning_rate' and 'subsample' replace 'min_samples_split'
}

# Grid search objects for both models
grid_search_rf = GridSearchCV(
    pipeline_rf,
    param_grid_rf,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search_xgb = GridSearchCV(
    pipeline_xgb,
    param_grid_xgb,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

In [9]:
from mlflow.tracking import MlflowClient

In [10]:
# Enable autologging so results are stored
mlflow.sklearn.autolog() # logs model params, metrics, model, sig(I/O schema), Artifacts ...
mlflow.xgboost.autolog() # logs model params, training/val metrics, feature importance, model itself

client = MlflowClient()
experiment_id = mlflow.get_experiment_by_name("Default").experiment_id

# Random Forest tuning
with mlflow.start_run(run_name="Random Forest Autolog Tuning") as rf_run:
    grid_search_rf.fit(X_train, y_train)
    best_rf = grid_search_rf.best_estimator_
    y_pred_rf = best_rf.predict_proba(X_test)[:, 1]
    test_auc_rf = roc_auc_score(y_test, y_pred_rf)
    mlflow.log_metric("test_roc_auc", test_auc_rf)   # common name
    rf_run_id = rf_run.info.run_id

# XGBoost tuning
with mlflow.start_run(run_name="XGBoost Autolog Tuning") as xgb_run:
    grid_search_xgb.fit(X_train, y_train)
    best_xgb = grid_search_xgb.best_estimator_
    y_pred_xgb = best_xgb.predict_proba(X_test)[:, 1]
    test_auc_xgb = roc_auc_score(y_test, y_pred_xgb)
    mlflow.log_metric("test_roc_auc", test_auc_xgb)   # common name
    xgb_run_id = xgb_run.info.run_id

Fitting 5 folds for each of 320 candidates, totalling 1600 fits


2026/03/30 13:27:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/30 13:27:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/30 13:27:52 INFO mlflow.sklearn.utils: Logging the 5 best runs, 315 runs will be omitted.


Fitting 5 folds for each of 54 candidates, totalling 270 fits


2026/03/30 13:27:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/30 13:27:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/30 13:27:56 INFO mlflow.sklearn.utils: Logging the 5 best runs, 49 runs will be omitted.


In [11]:
# Search all runs and sort by test_roc_auc best scores on top!
all_runs = client.search_runs(
    experiment_ids=[experiment_id],
    order_by=["metrics.test_roc_auc DESC"]
)

# print the name and score of the top 5 runs
for run in all_runs[:5]:
    run_name = run.data.tags.get("mlflow.runName", "Unnamed")
    score = run.data.metrics.get("test_roc_auc", 0)
    print(f"{run_name}: test_roc_auc = {score:.4f}")

Random Forest Autolog Tuning: test_roc_auc = 0.7936
Random Forest Autolog Tuning: test_roc_auc = 0.7885
Random Forest Autolog Tuning: test_roc_auc = 0.7885
Random Forest Autolog Tuning: test_roc_auc = 0.7827
Random Forest Autolog Tuning: test_roc_auc = 0.7827


In [12]:
#  defining function to get run id's so we can extract and use the model
def get_best_run_by_name(run_name):
    runs = client.search_runs(
        experiment_ids=[experiment_id],
        filter_string=f"tags.mlflow.runName = '{run_name}'",   # corrected
        order_by=["metrics.test_roc_auc DESC"],
        max_results=1
    )
    return runs[0] if runs else None

In [13]:
best_rf_run = get_best_run_by_name("Random Forest Autolog Tuning")
best_xgb_run = get_best_run_by_name("XGBoost Autolog Tuning")

if best_rf_run:
    print(f"Best RF: {best_rf_run.data.tags.get('mlflow.runName')} – test_roc_auc = {best_rf_run.data.metrics.get('test_roc_auc'):.4f}")
if best_xgb_run:
    print(f"Best XGB: {best_xgb_run.data.tags.get('mlflow.runName')} – test_roc_auc = {best_xgb_run.data.metrics.get('test_roc_auc'):.4f}")

Best RF: Random Forest Autolog Tuning – test_roc_auc = 0.7936
Best XGB: XGBoost Autolog Tuning – test_roc_auc = 0.7817


In [14]:
def get_best_run_by_tag(tag_value):
    runs = client.search_runs(
        experiment_ids=[experiment_id],
        filter_string=f"tags.model_type = '{tag_value}'",
        order_by=["metrics.test_roc_auc DESC"],
        max_results=1
    )
    return runs[0] if runs else None

In [15]:
model_rf = mlflow.sklearn.load_model(f"runs:/{best_rf_run.info.run_id}/model")
model_xgb = mlflow.sklearn.load_model(f"runs:/{best_xgb_run.info.run_id}/model")

In [16]:
# predicting with best model on test set!
y_pred_proba = model_rf.predict_proba(X_test)[:,1 ]
y_pred = model_rf.predict(X_test)

In [17]:
# Computing metrics
test_auc = roc_auc_score(y_test, y_pred_proba)
print(f"Test ROC-AUC: {test_auc: .4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Test ROC-AUC:  0.7936

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.57      0.67       150
           1       0.91      0.97      0.94       650

    accuracy                           0.90       800
   macro avg       0.87      0.77      0.81       800
weighted avg       0.89      0.90      0.89       800



In [18]:
# Since improvement was low, we are reviewing what parameters were chosen. 
print("Best parameters from grid search:")
print(grid_search_rf.best_params_)

Best parameters from grid search:
{'classifier__class_weight': None, 'classifier__max_depth': 20, 'classifier__min_samples_split': 10, 'classifier__n_estimators': 100}


In [19]:
from sklearn.metrics import precision_recall_fscore_support

y_pred = model_rf.predict(X_test)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary')
print(f"Precision: {precision:.3f}, Recall: {recall:.3f}, F1: {f1:.3f}")

Precision: 0.907, Recall: 0.974, F1: 0.939


### The above metrics are worse than the initial models.

In [20]:
# Get predicted probabilities for the test set
probs = model_rf.predict_proba(X_test)[:, 1]

# Sort by probability descending
sorted_indices = np.argsort(probs)[::-1]
sorted_y_true = y_test.iloc[sorted_indices].values if hasattr(y_test, 'iloc') else y_test[sorted_indices]

# Calculate cumulative opens captured at each percentile
cumulative_opens = np.cumsum(sorted_y_true)
percentiles = np.arange(1, len(sorted_y_true)+1) / len(sorted_y_true)

# Opens captured in top 30%
top30_cutoff = int(0.3 * len(sorted_y_true))
opens_in_top30 = cumulative_opens[top30_cutoff - 1]
total_opens = cumulative_opens[-1]
print(f"Top 30% captures {opens_in_top30}/{total_opens} = {opens_in_top30/total_opens:.2%} of all opens")

Top 30% captures 221/650 = 34.00% of all opens


### 34% is a marginal improvement on the 33.69% of captures from the baseline model.

## Further tuning
Additional XGB parameters include 'classifier__min_child_weight', 'classifier__gamma', etc., to the XGBoost grid for more thorough tuning. We skipped this as Random Forest out performed XGB and instead further tuned the Random Forest model to improve performance a bit.